# PROJECT 2: GRAM-SCHMIDT TO DECOMPOSE MATRIX INTO QR AND ITS APPLICATION

## I. STUDENT INFORMATION
* **Full Name:** Nguyễn Nhựt Huy
* **Student ID (MSSV):** 24127398
* **Class:** 24C06
* **Course:** Applied Mathematics and Statistics

## II. THEORY OVERVIEW

---

### 1. Gram-Schmidt Orthogonalization Theory
* **Objective:** To transform a set of linearly independent vectors $\{u_1, u_2, \dots, u_n\}$ into an orthonormal set $\{q_1, q_2, \dots, q_n\}$ where each $q_i$ is a unit vector and $\langle q_i, q_j \rangle = 0$ for $i \neq j$.
* **Algorithm:**
  1. Set $v_1 = u_1$.
  2. For $k = 2, \dots, n$: compute the projection of $u_k$ onto each previous $v_i$ and subtract:
     $$v_k = u_k - \sum_{i=1}^{k-1} \frac{\langle u_k, v_i \rangle}{\langle v_i, v_i \rangle} v_i$$
  3. Normalize each orthogonal vector: $q_k = \dfrac{v_k}{\|v_k\|}$.
* **Note:** The vectors $\{u_1, \dots, u_n\}$ must be linearly independent; otherwise the process will produce a zero vector at some step.

---

### 2. QR Decomposition Theory
* **Objective:** To factor any $m \times n$ matrix $A$ (with $m \geq n$ and full column rank) into the product of an $m \times n$ matrix $Q$ with orthonormal columns and an $n \times n$ upper triangular matrix $R$:
  $$A = Q R$$
* **Construction:** Apply the Gram-Schmidt process to the columns $u_1, u_2, \dots, u_n$ of $A$ to obtain orthonormal columns $q_1, q_2, \dots, q_n$ of $Q$. Then define $R$ as:
  $$R = \begin{pmatrix}
  \langle u_1, q_1 \rangle & \langle u_2, q_1 \rangle & \dots & \langle u_n, q_1 \rangle \\
  0 & \langle u_2, q_2 \rangle & \dots & \langle u_n, q_2 \rangle \\
  \vdots & \vdots & \ddots & \vdots \\
  0 & 0 & \dots & \langle u_n, q_n \rangle
  \end{pmatrix}$$
* **Properties:** $Q^T Q = I_n$ (the identity matrix), and $R$ is invertible when $A$ has full column rank.

---

### 3. Solving Linear Systems via QR Decomposition
* **Objective:** Solve the linear system $A x = b$ using the $QR$ factorization.
* **Derivation:**
  1. Substitute $A = QR$: $QRx = b$.
  2. Multiply both sides by $Q^T$: $Q^T Q R x = Q^T b$.
  3. Since $Q^T Q = I$, we obtain $R x = Q^T b$.
  4. $R$ is upper triangular, so the system can be solved by **back substitution**.
* **Key Insight:** Because $Q$ has orthonormal columns, its transpose equals its inverse: $Q^T = Q^{-1}$. This avoids computing the inverse explicitly and leverages the triangular structure of $R$ for efficient solving.
* **Applicability — Unique Solutions Only:**
  The QR-based approach $Rx = Q^T b$ requires the coefficient matrix $A$ to have **full column rank** (i.e., its columns are linearly independent). Under this condition:
  * $Q$ has orthonormal columns, so $Q^T Q = I$ holds exactly.
  * $R$ is an $n \times n$ upper triangular matrix that is **invertible** (all diagonal entries are non-zero).
  * The resulting system $Rx = Q^T b$ has a **unique solution** that can be obtained directly by back substitution.
* **Why not underdetermined or inconsistent systems?**
  * If $A$ is **rank-deficient** (columns are linearly dependent), the Gram-Schmidt process fails because some $v_k$ becomes the zero vector, and the decomposition returns $None$.
  * If $A$ is **underdetermined** (fewer rows than columns), the Gram-Schmidt process can still produce $Q$ and $R$, but $R$ will be singular (some diagonal entries are zero), making the back-substitution formula $x_j = (b_j - \sum) / R_{jj}$ impossible due to division by zero.
  * For **inconsistent systems**, the transformation $Rx = Q^T b$ may still produce a solution in the least-squares sense (when $m > n$), but the classical back-substitution routine would not detect inconsistency in the same way as Gauss elimination does — it simply reports a unique solution that minimizes $\|Ax - b\|$ rather than solving $Ax = b$ exactly.
* **In summary:** The $QR$ method described here is designed for **square or overdetermined** systems ($m \geq n$) with **full column rank**, guaranteeing a unique exact solution.

## III. IMPLEMENTATION

### Import necessary libraries

In [111]:
import math
import sympy as sp

### Support Functions

In [112]:
def dot_product(u, v):
    result = 0
    for i in range(len(u)):
        result += u[i] * v[i]
    return result

In [113]:
def is_zero_vector(u, tol=1e-9):
    for i in range(len(u)):
        if abs(u[i]) > tol:
            return False
    return True

In [114]:
def vector_norm(u):
    return math.sqrt(dot_product(u, u))

In [115]:
def multiply_cons_vector(u, k):
    return [k*v for v in u]

In [116]:
def get_col(u, i):
    return [u[j][i] for j in range(len(u))]

In [117]:
def sum_vector(u, v, plus = True):
    return [(m + n) if plus else (m - n) for m, n in zip(u, v)]

In [118]:
def transpose_matrix (A):
    rows, cols = len(A), len(A[0])
    return [[A[i][j] for i in range(rows)] for j in range(cols)]

In [119]:
def is_zero(val, tol=1e-9):
    return abs(val) < tol

In [120]:
def multiply_matrix_vector (A, u):
    if len(A[0]) != len(u):
        print("Cannot multiply")
        return None
    result = []
    for i in range(len(A)):
        temp = 0
        for j in range(len(u)):
            temp += A[i][j] * u[j]
        result.append(temp)
    return result

### QR decomposition

In [121]:
def qr_decomposition_Q(A, tol=1e-9):
    num_rows, num_cols = len(A), len(A[0])

    v_list = []
    q_list = []
    
    for i in range(num_cols):
        u_i = get_col(A, i)
        v_i = u_i.copy()

        for k in range(i):
            v_k = v_list[k]

            if is_zero_vector(v_k, tol):
                return None

            proj_coef = dot_product(u_i, v_k) / dot_product(v_k, v_k)
            temp = multiply_cons_vector(v_k, proj_coef)
            v_i = sum_vector(v_i, temp, False)
            
        v_list.append(v_i)

        norm_v_i = vector_norm(v_i)

        if norm_v_i < tol:
            return None
        else:
            q_i = multiply_cons_vector(v_i, 1.0 / norm_v_i)
            
        q_list.append(q_i)

    Q = transpose_matrix(q_list)

    R = [[0.0 for _ in range(num_cols)] for _ in range(num_cols)]

    for i in range (num_cols):
        for j in range (i, num_cols):
            u_j = get_col(A, j)
            R[i][j] = dot_product(u_j, q_list[i]) 

    return [Q, R]

### Solving Linear Systems using QR Decomposition

In [122]:
def back_substitution(A):
    n, m = [len(A), len(A[0])]
    num_variables = m - 1
    for i in range(n-1, -1, -1):
        all_variables_zero = all([is_zero(A[i][j]) for j in range(num_variables)])
        left_zero = not is_zero(A[i][m-1])
        if all_variables_zero and left_zero:
            return None
    x_sym = [sp.symbols(f'x_{i+1}') for i in range(num_variables)]
    pivot_columns = []
    for i in range(n):
        for j in range(num_variables):
            if not is_zero(A[i][j]):
                pivot_columns.append(j)
                break
    num_pivots = len(pivot_columns)
    for i in range(num_pivots - 1, -1, -1):
        j = pivot_columns[i]
        left_sum = sum([A[i][k] * x_sym[k] for k in range(j+1, num_variables)])
        x_sym[j] = sp.simplify((A[i][m-1] - left_sum) / A[i][j])
    return x_sym

In [123]:
def solve_using_QR(A, b):
    m, n = len(A), len(A[0])

    if m < n:
        return None

    result = qr_decomposition_Q(A)
    if result is None:
        return None

    Q, R_mat = result
    Qt = transpose_matrix(Q)
    b2 = multiply_matrix_vector(Qt, b)
    augmented = [R_mat[i] + [b2[i]] for i in range(len(R_mat))]
    x = back_substitution(augmented)

    if x is None:
        return None
    
    x = [float(val) for val in x]

    # Overdetermined: verify the solution satisfies all equations
    if m > n:
        x_vals = [float(v) for v in x]
        Ax = multiply_matrix_vector(A, x_vals)
        residual = math.sqrt(sum((Ax[i] - b[i])**2 for i in range(m)))
        if residual > 1e-9:
            return None

    return x

## IV. TEST CASES

### QR Decomposition Test Cases

In [124]:
qr_test_cases = [
    [[1, 1, 2],
     [2, -1, 1],
     [-2, 4, 1]],

    [[1, 1, 1],
     [2, -2, 2],
     [1, 1, -1]],

    [[1, 1, -1],
     [0, 1, 2],
     [1, 1, 1]],

    [[-1, -1, 1],
     [1, 3, 3],
     [-1, -1, 5],
     [1, 3, 7]],

    [[1, 1, 1],
     [2, 2, 0],
     [3, 0, 0],
     [0, 0, 1]],

    [[-2, 1, 3],
     [1, 0, 0],
     [0, 1, 0],
     [0, 0, 1]],

    [[1, -1, 2],
     [1, 0, -1],
     [-1, 1, 2],
     [0, 1, 1]]
]

In [125]:
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g']
for idx, A in enumerate(qr_test_cases):
    print(f"{'='*50}")
    print(f"Test case {labels[idx]}:")
    print(f"{'='*50}")
    print("Matrix A:")
    for row in A:
        print(row)

    result = qr_decomposition_Q(A)
    if result is None:
        print("QR decomposition failed: columns are linearly dependent")
    else:
        Q, R = result
        print("\nMatrix Q:")
        for row in Q:
            print([round(v, 6) for v in row])
        print("\nMatrix R:")
        for row in R:
            print([round(v, 6) for v in row])

        # Verify Q^T Q = I
        Qt = transpose_matrix(Q)
        QtQ = [[0.0]*len(Q[0]) for _ in range(len(Qt))]
        for i in range(len(Qt)):
            for j in range(len(Q[0])):
                for k in range(len(Q)):
                    QtQ[i][j] += Qt[i][k] * Q[k][j]
        print("\nQ^T * Q (should be identity):")
        for row in QtQ:
            print([round(v, 6) for v in row])

Test case a:
Matrix A:
[1, 1, 2]
[2, -1, 1]
[-2, 4, 1]

Matrix Q:
[0.333333, 0.666667, 0.666667]
[0.666667, 0.333333, -0.666667]
[-0.666667, 0.666667, -0.333333]

Matrix R:
[3.0, -3.0, 0.666667]
[0.0, 3.0, 2.333333]
[0.0, 0.0, 0.333333]

Q^T * Q (should be identity):
[1.0, 0.0, 0.0]
[0.0, 1.0, -0.0]
[0.0, -0.0, 1.0]
Test case b:
Matrix A:
[1, 1, 1]
[2, -2, 2]
[1, 1, -1]

Matrix Q:
[0.408248, 0.57735, 0.707107]
[0.816497, -0.57735, 0.0]
[0.408248, 0.57735, -0.707107]

Matrix R:
[2.44949, -0.816497, 1.632993]
[0.0, 2.309401, -1.154701]
[0.0, 0.0, 1.414214]

Q^T * Q (should be identity):
[1.0, -0.0, 0.0]
[-0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]
Test case c:
Matrix A:
[1, 1, -1]
[0, 1, 2]
[1, 1, 1]

Matrix Q:
[0.707107, 0.0, -0.707107]
[0.0, 1.0, 0.0]
[0.707107, 0.0, 0.707107]

Matrix R:
[1.414214, 1.414214, 0.0]
[0.0, 1.0, 2.0]
[0.0, 0.0, 1.414214]

Q^T * Q (should be identity):
[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]
Test case d:
Matrix A:
[-1, -1, 1]
[1, 3, 3]
[-1, -1, 5]
[1, 3, 7]

Mat

### Solving Linear Systems using QR - Test Cases

In [ ]:
solve_test_cases = [
    # --- Unique solution (determined square) ---
    [[1, 2, -1, -1],
     [2, 2, 1, 1],
     [3, 5, -2, -1]],

    [[1, 2, 0, 2, 6],
     [3, 5, -1, 6, 17],
     [2, 4, 1, 2, 12],
     [2, 0, -7, 11, 7]],

    [[2, -4, 6, 8],
     [1, -1, 1, -1],
     [1, -3, 4, 0]],

    [[1, -2, 3, -3],
     [2, 2, 0, 0],
     [0, -3, 4, 1],
     [1, 0, 1, -1]],

    # --- Overdetermined consistent (exact solution exists) ---
    [[1, 2, 5],
     [3, 4, 11],
     [1, 0, 1]],

    # --- Overdetermined inconsistent (no exact solution) ---
    [[1, 2, 5],
     [3, 4, 11],
     [0, 1, 1]],

    # --- Underdetermined consistent (infinitely many solutions) ---
    [[1, 1, 0, 3],
     [0, 1, 1, 4]],

    # --- Underdetermined inconsistent (no solution) ---
    [[1, 1, 0, 3],
     [1, 1, 0, 5]]
]

In [127]:
for idx, aug in enumerate(solve_test_cases, start=1):
    print(f"{'='*50}")
    print(f"Solving System (Test {idx}):")
    print(f"{'='*50}")

    A_coef = [row[:-1] for row in aug]
    b_vec = [row[-1] for row in aug]

    print("Coefficient matrix A:")
    for row in A_coef:
        print(row)
    print("\nRight-hand side b:")
    print(b_vec)

    solution = solve_using_QR(A_coef, b_vec)

    print("\n--- [QR Method Result] ---")
    
    if solution is None:
        if len(A_coef) < len(A_coef[0]):
            print("Status: Underdetermined System (Cannot solve using QR method)")
        elif len(A_coef) > len(A_coef[0]):
            # Vì hàm của bạn return None khi residual > 1e-9, nên nhảy vào đây 
            # nghĩa là hệ quá định không có nghiệm chính xác (No Exact Solution)
            print("Status: Overdetermined System (No Exact Solution / Cannot solve using QR method)")
        else:
            print("Status: Cannot solve using QR method")

    else:
        if len(A_coef) > len(A_coef[0]):
            # Thỏa mãn điều kiện m > n và residual <= 1e-9
            print("Status: Overdetermined System (Consistent with Exact Solution)")
        else:
            # Hệ vuông thông thường m == n
            print("Status: Consistent System (Unique Solution)")
            
        print("Solution values:")
        for i, val in enumerate(solution):
            float_val = float(val)
            formatted_val = int(float_val) if float_val.is_integer() else round(float_val, 4)
            print(f"   x_{i+1} = {formatted_val}")

Solving System (Test 1):
Coefficient matrix A:
[1, 2, -1]
[2, 2, 1]
[3, 5, -2]

Right-hand side b:
[-1, 1, -1]

--- [QR Method Result] ---
Status: Consistent System (Unique Solution)
Solution values:
   x_1 = 4.0
   x_2 = -3.0
   x_3 = -1.0
Solving System (Test 2):
Coefficient matrix A:
[1, 2, 0, 2]
[3, 5, -1, 6]
[2, 4, 1, 2]
[2, 0, -7, 11]

Right-hand side b:
[6, 17, 12, 7]

--- [QR Method Result] ---
Status: Consistent System (Unique Solution)
Solution values:
   x_1 = 2.0
   x_2 = 3.0
   x_3 = -2.0
   x_4 = -1.0
Solving System (Test 3):
Coefficient matrix A:
[2, -4, 6]
[1, -1, 1]
[1, -3, 4]

Right-hand side b:
[8, -1, 0]

--- [QR Method Result] ---
Status: Consistent System (Unique Solution)
Solution values:
   x_1 = 3.0
   x_2 = 13.0
   x_3 = 9.0
Solving System (Test 4):
Coefficient matrix A:
[1, -2, 3]
[2, 2, 0]
[0, -3, 4]
[1, 0, 1]

Right-hand side b:
[-3, 0, 1, -1]

--- [QR Method Result] ---
Status: Overdetermined System (Consistent with Exact Solution)
Solution values:
   x_1 

## V. IMPLEMENTATION IDEA AND FUNCTION DESCRIPTIONS

---

### 1. Code Architecture & Implementation Strategy

#### A. Data Models and Precision Control
* **List-of-Lists Representation:** Matrices are represented as Python lists of lists (each inner list is a row). This keeps the implementation self-contained without heavy library dependencies for the core algorithm.
* **Epsilon Thresholding:** To guard against floating-point artifacts, an explicit tolerance parameter ($tol = 10^{-9}$) is used when checking for zero vectors. A vector is considered zero if all its components fall below this threshold.

#### B. Program Execution Workflow
1. **Gram-Schmidt Orthogonalization:** The algorithm processes each column of $A$ sequentially, orthogonalizing against previously processed columns and finally normalizing to obtain the orthonormal columns of $Q$.
2. **R Matrix Construction:** Once $Q$ is known, $R$ is filled by computing dot products $R[i][j] = \langle u_j, q_i \rangle$ for $i \leq j$.
3. **Solving via QR:** For a system $Ax = b$, the QR factors are computed, then $b$ is transformed to $Q^T b$, and finally back substitution solves the upper-triangular system $Rx = Q^T b$.

---

### 2. Detailed Function Descriptions

#### A. Helper Subroutines

| Function Name & Signature | Programmatic Utility | Input Parameters | Return Value |
| :--- | :--- | :--- | :--- |
| `dot_product(u, v)` | Computes the Euclidean inner product of two vectors $\sum u_i v_i$. | <ul><li>`u` (list): First vector</li><li>`v` (list): Second vector</li></ul> | `float`: The scalar dot product. |
| `is_zero_vector(u, tol)` | Tests whether a vector is the zero vector within a tolerance. | <ul><li>`u` (list): Vector to check</li><li>`tol` (float): Tolerance (Default `1e-9`)</li></ul> | `bool`: `True` if all entries are below `tol`. |
| `vector_norm(u)` | Returns the Euclidean norm $\|u\| = \sqrt{\langle u, u \rangle}$. | <ul><li>`u` (list): Input vector</li></ul> | `float`: The norm. |
| `multiply_cons_vector(u, k)` | Scales a vector by a scalar $k$. | <ul><li>`u` (list): Input vector</li><li>`k` (float): Scalar factor</li></ul> | `list`: The scaled vector. |
| `get_col(A, i)` | Extracts the $i$-th column from a matrix stored as a list of rows. | <ul><li>`A` (list of list): Matrix</li><li>`i` (int): Column index</li></ul> | `list`: The column vector. |
| `sum_vector(u, v, plus)` | Adds or subtracts two vectors element-wise. | <ul><li>`u`, `v` (list): Vectors</li><li>`plus` (bool): `True` for addition, `False` for subtraction</li></ul> | `list`: Resultant vector. |
| `transpose_matrix(A)` | Transposes a matrix (rows become columns). | <ul><li>`A` (list of list): Input matrix</li></ul> | `list of list`: Transposed matrix. |
| `multiply_matrix_vector(A, u)` | Multiplies a matrix by a column vector. | <ul><li>`A` (list of list): Matrix</li><li>`u` (list): Vector</li></ul> | `list`: Product vector. |
| `is_zero(val, tol)` | Tests if a scalar is within tolerance of zero. | <ul><li>`val` (float): Value to test</li><li>`tol` (float): Tolerance</li></ul> | `bool`: `True` if $|val| < tol$. |

#### B. Core Solving Engines

### `qr_decomposition_Q(A, tol=1e-9)`
* **Functional Description:** Performs the Gram-Schmidt process on the columns of $A$ to produce an orthonormal matrix $Q$ and an upper-triangular matrix $R$ such that $A = QR$.
* **Input:** `A` (list of list): An $m \times n$ matrix with $m \geq n$ and full column rank.
* **Output:** `[Q, R]` where `Q` ($m \times n$) has orthonormal columns and `R` ($n \times n$) is upper triangular. Returns `None` if the columns are linearly dependent.

### `back_substitution(A)`
* **Functional Description:** Solves an upper-triangular augmented system $[R | b]$ by backward substitution, supporting unique and infinitely many solutions via sympy symbolic expressions.
* **Input:** `A` (numpy.ndarray): An $n \times (n+1)$ augmented matrix in row-echelon form.
* **Output:** `list` of solution values or symbolic expressions; `None` if the system is inconsistent.

### `solve_using_QR(A, b)`
* **Functional Description:** Combines $QR$ decomposition and back substitution to solve the linear system $Ax = b$. Computes $Q$ and $R$, then solves $Rx = Q^T b$.
* **Input:** <ul><li>`A` (list of list): $m \times n$ coefficient matrix</li><li>`b` (list): Right-hand side vector of length $m$</li></ul>
* **Output:** `list` of solution values/symbolic expressions; `None` if decomposition fails or system is inconsistent.